# Stage 1-2 FITS-PSF evaluation
Uses `model.eval()` and the same FITS PSF operator as training.


In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import data
from differentiable_lensing import DifferentiableLensing
from psf import apply_psf, build_psf_kernel
from sisr import SISR
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHTS_PATH=Path('hsc_fits_psf_weights.pt')
PSF_PATH=Path('path/hsc.fits')
RESOLUTION=0.168
PSF_SOURCE_PIXSCALE=0.168
TARGET_RESOLUTION=RESOLUTION/2
TARGET_SHAPE=128


In [ ]:
model=SISR(2,1,3,in_channels=2,latent_channel_count=64).to(DEVICE)
loaded=torch.load(WEIGHTS_PATH,map_location=DEVICE)
model.load_state_dict(loaded.get('model_state_dict',loaded) if isinstance(loaded,dict) else loaded)
model.eval()
lens=DifferentiableLensing(device=DEVICE,alpha=None,target_resolution=TARGET_RESOLUTION,target_shape=TARGET_SHAPE).to(DEVICE)
forward=[torch.load('scatter_to_log_128.pt',map_location=DEVICE).to(DEVICE),torch.load('forward_from_log_128.pt',map_location=DEVICE).to(DEVICE),torch.load('scatter_from_log_128.pt',map_location=DEVICE).to(DEVICE)]
backward=torch.load('sparse_grid_fracs_euclid_backward.pt',map_location=DEVICE).to(DEVICE)
psf=build_psf_kernel('fits',0.16,TARGET_RESOLUTION,path=str(PSF_PATH),source_pixscale_arcsec=PSF_SOURCE_PIXSCALE,device=DEVICE)


In [ ]:
lr=data.LensingDataset('val/',['no_sub'],2000)[0].unsqueeze(0).float().to(DEVICE)
lr=lr.unsqueeze(1) if lr.ndim==3 else lr
with torch.inference_mode():
    source_lr=lens.cross_grid_fill(lr,[backward])
    source_hr=model(torch.cat([source_lr,lr],dim=1))
    intrinsic=lens.cross_grid_fill(source_hr,forward)
    pred=F.interpolate(apply_psf(intrinsic,psf),size=lr.shape[-2:],mode='area')
zero_mse=lr.square().mean(); model_mse=F.mse_loss(pred,lr)
print('zero MSE',float(zero_mse),'model MSE',float(model_mse),'skill',float(1-model_mse/zero_mse.clamp_min(1e-8)))
fig,ax=plt.subplots(1,3,figsize=(12,4))
for a,t,x in zip(ax,['LR','prediction','residual'],[lr[0,0],pred[0,0],pred[0,0]-lr[0,0]]):
    a.imshow(x.cpu(),origin='lower',cmap='gray'); a.set_title(t)
plt.show()
